# Normalización de datos y dataset de notas comerciales

Este notebook es el punto de entrada para **generar tú mismo los archivos de `normalized/`**. Ejecuta las celdas de arriba hacia abajo con un kernel Python 3.10 o superior.

- Lee `data/usuarios.csv`, `equipos.csv`, `registros.csv`, `ausencias.csv` y `actividad.csv`.
- Conserva los originales y genera valores normalizados, reglas de transformación e incidencias.
- Reconstruye propiedad histórica y carga de trabajo inferida, sin asignar vendedores.
- Genera referencias, entrenamiento y evaluación de notas a partir del prompt v1 aprobado.
- La inferencia con Ollama es **opcional y está desactivada por defecto**. No se descarga ningún modelo ni se inicia Docker al ejecutar todas las celdas.

**Aprobación:** el usuario validó el prompt v1 el 2026-09-18 y autorizó continuar. Ver `normalized/approval_note.md`. Aprobar el prompt no significa que el modelo haya superado la evaluación. Las ejecuciones anteriores se detuvieron al cambiar a este flujo con notebook; sus respuestas parciales se conservan sin declararlas aceptadas.

Si ya existen resultados, las celdas de generación reemplazan únicamente sus archivos generados. Los CSV originales, el golden revisado y el historial de evaluación se conservan.


## 1. Preparar el kernel

Instala las dependencias en el mismo entorno del kernel, desde la raíz del repositorio:

```bash
python3 -m pip install -r requirements.txt
```

Abre `normalization_workflow.ipynb` con Jupyter o VS Code y selecciona ese entorno. No se requiere pandas.


In [ ]:
import os
import sys
import json
import csv
import hashlib
import html
from pathlib import Path
from datetime import date
from collections import Counter
from IPython.display import display, HTML, Markdown, FileLink

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "normalization" / "pipeline.py").is_file()
     and (p / "data" / "promp_normalization.md").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Abre este notebook desde la carpeta del repositorio.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from normalization import pipeline, notes
from normalization.common import read_csv, read_jsonl, digest
from normalization.evaluate_note_prompt import evaluate

OUTPUT_DIR = ROOT / "normalized"
print("Repositorio:", ROOT)
print("Salida:", OUTPUT_DIR)


## 2. Configuración de normalización

La fecha efectiva determina fechas futuras y ausencias vigentes. Una zona vacía permanece vacía salvo que agregues una regla explícita de ciudad. Cambiar la fecha puede producir diferencias legítimas con los conteos de referencia del 2026-09-18.


In [ ]:
INPUT_DIR = ROOT / "data"
EFFECTIVE_DATE = date(2026, 9, 18)
CITY_TO_ZONE = {}  # Ejemplo explícito: {"bogotá": "Centro"}
APPROVED_PROMPT = ROOT / "project/llm_service/promps/note_extraction_v1.md"
RUN_TESTS = True

TABLE_NAMES = ("usuarios", "equipos", "registros", "ausencias", "actividad")
SOURCE_PATHS = {name: INPUT_DIR / f"{name}.csv" for name in TABLE_NAMES}
missing = [str(path) for path in SOURCE_PATHS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Faltan CSV: " + ", ".join(missing))
SOURCE_HASHES_BEFORE = {name: digest(path) for name, path in SOURCE_PATHS.items()}
print("Fecha efectiva:", EFFECTIVE_DATE)
print("Reglas de ciudad:", CITY_TO_ZONE or "Ninguna")


In [ ]:
def show_table(rows, columns=None, limit=12):
    """Tabla de lectura; escapa el contenido de los CSV antes de mostrar HTML."""
    rows = list(rows)
    if not rows:
        print("Sin filas.")
        return
    columns = columns or list(rows[0])
    def text(value):
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        return html.escape("null" if value is None else str(value))
    header = "".join(f"<th>{text(c)}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{text(row.get(c))}</td>" for c in columns) + "</tr>"
        for row in rows[:limit]
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    print(f"Mostrando {min(limit, len(rows))} de {len(rows)} filas.")

source_tables = {name: read_csv(path) for name, path in SOURCE_PATHS.items()}
show_table([
    {"tabla": name, "filas": len(rows), "columnas": list(rows[0]) if rows else []}
    for name, rows in source_tables.items()
])
show_table(source_tables["registros"],
           ["id", "razon_social", "zona", "estado", "notas"], limit=5)


## 3. Generar los CSV normalizados

**Esta celda escribe en `normalized/`.** Cada campo conserva el original, su valor normalizado, la regla aplicada y si hubo inferencia. No elimina filas ni fusiona duplicados. En CSV, los valores nulos se representan como celdas vacías; en JSON, como `null`.

Propiedad y carga son inferidas a partir de la actividad: no prueban quién tiene actualmente la cuenta ni que la asignación histórica fuera óptima. La carga cuenta registros `asignado`/`en_gestion`, no actividades. Capacidad nula y capacidad cero tienen significados distintos.


In [ ]:
report = pipeline.run(
    source=INPUT_DIR,
    effective=EFFECTIVE_DATE,
    city_rules=CITY_TO_ZONE,
)
show_table([
    {"control": name, **check}
    for name, check in report["checks"].items()
], limit=30)
discrepancies = {name: check for name, check in report["checks"].items() if not check["matches"]}
print("Discrepancias para investigar:", discrepancies or "Ninguna")
print("Filas conservadas:", report["row_counts"])


## 4. Revisar calidad, duplicados y carga

Consulta el detalle de incidencias antes de consumir los datos. Una referencia inválida o una capacidad indefinida no se corrige mediante una suposición silenciosa.


In [ ]:
issues = read_csv(OUTPUT_DIR / "data_quality_issues.csv")
show_table([
    {"tipo": name, "cantidad": count}
    for name, count in Counter(row["issue_type"] for row in issues).items()
])
show_table(issues, ["table_name", "row_id", "field_name", "issue_type", "original_value", "is_blocking"])
show_table(read_csv(OUTPUT_DIR / "duplicate_groups.csv"))
show_table(read_csv(OUTPUT_DIR / "seller_workload.csv"),
           ["seller_id", "assigned_count", "in_management_count", "open_workload",
            "maximum_capacity", "remaining_capacity", "availability_status", "eligibility_reasons"],
           limit=20)
print("Registros activos sin propietario atribuible:", report["unattributed_active_records"])


## 5. Revisar el prompt aprobado y las referencias

Se usan las 15 interpretaciones revisadas. La petición de experiencia sectorial requiere revisión adicional cuando falta `sector`; no se inventa ese dato.

El prompt aprobado permanece intacto. Para experimentar, crea `note_extraction_v2.md` u otra versión y úsala en la sección de evaluación, sin sobrescribir v1 ni sustituir automáticamente el prompt aprobado de entrenamiento.


In [ ]:
notes.ensure_review_artifacts()
approval = json.loads((OUTPUT_DIR / "prompt_review_status.json").read_text(encoding="utf-8"))
assert approval["status"] == "approved", "Falta aprobación del prompt."
assert digest(APPROVED_PROMPT) == approval["approved_prompt_sha256"], "El prompt aprobado cambió. Usa una versión nueva."
golden = read_jsonl(OUTPUT_DIR / "note_golden_review_v1.jsonl")
print(approval["approval_note"])
show_table([
    {"plantilla": item["note_template_id"], "nota": item["raw_note"],
     "acción": item["expected_output"]["action"],
     "códigos": item["expected_output"]["reason_codes"], "revisión": item["review_status"]}
    for item in golden
], limit=15)
display(FileLink(str(APPROVED_PROMPT.relative_to(ROOT))))


## 6. Generar datasets de notas

Esta celda produce:

- `note_examples.csv`: una fila por registro, incluidos los registros sin notas.
- `note_examples.jsonl`: una referencia por texto original único, con frecuencia y IDs.
- `note_training.jsonl`: mensajes de entrenamiento sin duplicar textos para aumentar artificialmente el tamaño.
- `note_evaluation.jsonl`: entradas y resultados esperados; el evaluador solo envía la entrada al modelo.
- `note_holdout.jsonl`: paráfrasis sintéticas, separadas del entrenamiento y los ejemplos del prompt.
- `note_security_evaluation.jsonl` y `note_context_evaluation.jsonl`: pruebas adicionales de inyección y contexto.

Las paráfrasis fueron revisadas por el agente contra las interpretaciones aprobadas; no se atribuye revisión humana independiente.


In [ ]:
dataset_counts = notes.build(prompt=APPROVED_PROMPT)
print(dataset_counts)
show_table(read_csv(OUTPUT_DIR / "note_examples.csv"),
           ["record_id", "raw_note", "has_note", "note_template_id", "review_status"], limit=8)
show_table(read_jsonl(OUTPUT_DIR / "note_holdout.jsonl"),
           ["note_template_id", "raw_note", "source", "review_status"], limit=15)


## 7. Ejecutar pruebas automatizadas

Verifican idempotencia, alias, nulos, duplicados, referencias, fechas, ausencias, propiedad, carga, esquema y separación de datasets. Las pruebas de transporte usan respuestas simuladas: **no son evidencia de precisión real del LLM**.


In [ ]:
import subprocess

if RUN_TESTS:
    completed = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
        cwd=ROOT, capture_output=True, text=True,
    )
    print(completed.stdout + completed.stderr)
    completed.check_returncode()
else:
    print("Pruebas omitidas por configuración.")


## 8. Evaluación opcional con Ollama

La generación de datos ya está terminada. Solo activa las opciones siguientes cuando quieras ejecutar inferencias reales.

Si necesitas arrancar el servicio y descargar un modelo, ejecuta en una terminal desde la raíz del repositorio:

```bash
docker compose -f project/docker-compose.yml up -d llm_service
docker compose -f project/docker-compose.yml exec -T llm_service ollama pull llama3.2:1b
```

Configura `LLM_API_URL`, `LLM_MODEL` y `LLM_TIMEOUT_SECONDS` en tu entorno o en la celda siguiente. Por ejemplo, URL base `http://localhost:11434` y modelo `llama3.2:1b`. Este modelo sirve para probar el flujo: **no está aceptado para uso operativo**.

La evaluación de 15 notas puede tardar varios minutos en CPU. Cada ejecución conserva prompt, esquema, entradas y respuestas en un directorio único de `normalized/evaluation_runs/`. Un fallo o interrupción no se presenta como aceptación.


In [ ]:
LLM_API_URL = os.environ.get("LLM_API_URL", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "")
LLM_TIMEOUT_SECONDS = float(os.environ.get("LLM_TIMEOUT_SECONDS", "600"))
EVALUATION_PROMPT = APPROVED_PROMPT

EVALUATE_ORIGINALS = False
EVALUATE_HOLDOUT = False
EVALUATE_SECURITY = False
EVALUATE_CONTEXT = False

print("Prompt de evaluación:", EVALUATION_PROMPT.name)
print("Modelo:", LLM_MODEL or "sin configurar")
print("Evaluación habilitada:", any([
    EVALUATE_ORIGINALS, EVALUATE_HOLDOUT, EVALUATE_SECURITY, EVALUATE_CONTEXT
]))


In [ ]:
evaluation_jobs = [
    (EVALUATE_ORIGINALS, "note_evaluation.jsonl", "note_extraction_results.csv"),
    (EVALUATE_HOLDOUT, "note_holdout.jsonl", "note_extraction_results_holdout.csv"),
    (EVALUATE_SECURITY, "note_security_evaluation.jsonl", "note_extraction_results_security.csv"),
    (EVALUATE_CONTEXT, "note_context_evaluation.jsonl", "note_extraction_results_context.csv"),
]
current_summaries = []
if any(enabled for enabled, _, _ in evaluation_jobs):
    if not LLM_API_URL or not LLM_MODEL:
        raise ValueError("Configura LLM_API_URL y LLM_MODEL antes de evaluar.")
    for enabled, examples_name, results_name in evaluation_jobs:
        if enabled:
            summary = evaluate(
                EVALUATION_PROMPT, OUTPUT_DIR / examples_name, OUTPUT_DIR / results_name,
                LLM_API_URL, LLM_MODEL, LLM_TIMEOUT_SECONDS,
            )
            current_summaries.append(summary)
else:
    print("No se hicieron llamadas al LLM. Activa las opciones cuando quieras evaluar.")

for summary in current_summaries:
    print(summary["evaluation_set"], summary["metrics"])
    print("Cumple todos los umbrales:", summary["acceptance_thresholds_satisfied"])


## 9. Interpretar las métricas y crear nuevas versiones

Umbrales exigidos para las notas originales: JSON y esquema válidos 100%; recall de no contactar y posible duplicado 100%; acción exacta 100%; precisión por campo ≥95%; cero afirmaciones inventadas detectadas contra la referencia.

Las 15 notas originales son un conjunto de **desarrollo**, ya conocido por el prompt. No demuestran generalización. Lee por separado los resultados del holdout sintético, de inyección y de contexto. Si un conjunto no contiene positivos para un recall, el evaluador devuelve `null` y no declara cumplidos todos los umbrales globales.

Si hay errores, revisa `field_mismatches` y `unsupported_fields`, crea una nueva versión del prompt que corrija únicamente los problemas observados, cambia `EVALUATION_PROMPT` y repite el conjunto completo. No incorpores las paráfrasis reservadas a los ejemplos del prompt. Conserva los fallos y no presentes un modelo como aceptado mientras no cumpla los umbrales.


In [ ]:
for summary_path in sorted(OUTPUT_DIR.glob("note_extraction_results*.summary.json")):
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(summary_path.name, summary["evaluation_set"], summary["metrics"])

partial_runs = [
    p for p in (OUTPUT_DIR / "evaluation_runs").glob("*")
    if p.is_dir() and not (p / "summary.json").exists()
]
print("Ejecuciones parciales, sin conclusión de aceptación:", len(partial_runs))


## 10. Comprobar originales y abrir los archivos generados

Los artefactos persistentes están en la carpeta `normalized/` del repositorio. El notebook se entrega sin salidas ejecutadas para que puedas generar y revisar tus propios resultados.


In [ ]:
SOURCE_HASHES_AFTER = {name: digest(path) for name, path in SOURCE_PATHS.items()}
assert SOURCE_HASHES_BEFORE == SOURCE_HASHES_AFTER, "Los CSV originales cambiaron durante la sesión."
print("Los cinco CSV originales permanecen intactos.")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file() and path.suffix in {".csv", ".json", ".jsonl", ".md"}:
        display(FileLink(str(path.relative_to(ROOT))))
